In [1]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", False)
import os, json, numpy as np, matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts/experiments_large_scale/')

import run_LowRank

from run_LowRank import (seed_everything, get_device,
            stratified_equal_halves
)

In [2]:
"""
Timepoints, indexed by AnnData index
"""
timepoints_across_datasets =  [['E8.5', 'E8.75', 'E9.0', 'E9.25', 'E9.5', 'E9.75', 'E10.0']]
'''
timepoints_across_datasets =  [['E8.5',
                                'E9.0',
                                'E9.5',
                                'E10.0',
                                'E10.5']] #, 'E11.5', 'E12.5']]'''

'''
timepoints_across_datasets =  [['E8.5', 'E8.75', 'E9.0', 'E9.25', 'E9.5', 'E9.75', 'E10.0', 'E10.25', 'E10.5', 'E10.75'], \
                               ['E11.0', 'E11.25', 'E11.5', 'E11.75', 'E12.0', 'E12.25', 'E12.5', 'E12.75', 'E13.0', 'E13.25', 'E13.5', 'E13.75'], \
                               ['E14.0', 'E14.25', 'E14.333', 'E14.75', 'E15.0', 'E15.25', 'E15.5', 'E15.75', 'E16.0', 'E16.25', 'E16.5', 'E16.75'], \
                               ['E17.0', 'E17.25', 'E17.5', 'E17.75', 'E18.0', 'E18.25', 'E18.5', 'E18.75']]
'''

"""
For simplicity, we choose the first replicate for each timepoint as our dataset.
"""
replicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30']]
'''
replicates_across_datasets = [['embryo_11', 
                               'embryo_16',
                               'embryo_24',
                               'embryo_30',
                               'embryo_34']] #, 'embryo_38', 'embryo_42']]'''
'''
replicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'], \
                              ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'], \
                              ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'], \
                              ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]
'''


"\nreplicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'],                               ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'],                               ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'],                               ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]\n"

In [3]:
import os, json, sys, gc
import numpy as np
import pandas as pd
import scanpy as sc
import single_cell
import importlib
import monge_rotate_lr

importlib.reload(single_cell)
importlib.reload(run_LowRank)
importlib.reload(monge_rotate_lr)

# Cell-type labels (index must contain 'cell_id' to join on)
df_cell = pd.read_csv("/scratch/gpfs/ph3641/hm_ot/df_cell.csv").set_index("cell_id")
# Whether minor subsampling is needed -- for HiRef to work need many divisors, so decrease n slightly to get this
subsample_to_nonprime_tp = [False, True, True, True, True, True, True]
max_Qs = [1000, 100, 100, 100, 100, 100, 100]
max_ranks=[100, 5000, 5000, 5000, 5000, 5000, 5000]

all_results = []

for i in range(len(timepoints_across_datasets)):
    # --- load dataset i ---
    filehandle_ME = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+1}.h5ad"
    adata = sc.read_h5ad(filehandle_ME, backed="r")

    timepoints = timepoints_across_datasets[i]
    replicates = replicates_across_datasets[i]

    for tp in range(len(timepoints)):
        subset_adata = None
        
        if (tp + 1) < len(timepoints):
            # pair within the same dataset
            t1, t2 = timepoints[tp], timepoints[tp + 1]
            r1, r2 = replicates[tp], replicates[tp + 1]
            print(f"[intra] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # subset by original string fields; move to memory
            subset_adata = adata[
                (adata.obs["day"].isin([t1, t2])) &
                (adata.obs["embryo_id"].isin([r1, r2]))
            ].to_memory()

        elif (i + 1) < len(timepoints_across_datasets):
            # cross to the next dataset's first timepoint
            timepoints2 = timepoints_across_datasets[i + 1]
            replicates2 = replicates_across_datasets[i + 1]
            t1, t2 = timepoints[tp], timepoints2[0]
            r1, r2 = replicates[tp], replicates2[0]
            print(f"[inter] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # last slice from current dataset to memory
            adata1 = adata[
                (adata.obs["day"] == t1) &
                (adata.obs["embryo_id"] == r1)
            ].to_memory()

            # open next dataset and slice first pair to memory
            filehandle_ME2 = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+2}.h5ad"
            adata2_full = sc.read_h5ad(filehandle_ME2, backed="r")
            adata2 = adata2_full[
                (adata2_full.obs["day"] == t2) &
                (adata2_full.obs["embryo_id"] == r2)
            ].to_memory()
            del adata2_full; gc.collect()

            # concatenate the two to one in-memory object
            subset_adata = adata1.concatenate(adata2, index_unique=None)
            del adata1, adata2; gc.collect()
        else:
            # nothing to align at the tail
            continue

        # skip if empty (can happen with missing replicates)
        if subset_adata.n_obs == 0:
            print("  -> empty subset, skipping")
            continue

        # integrate labels (expects 'cell_id' present as a column)
        if "cell_id" not in subset_adata.obs.columns:
            print("  -> missing 'cell_id' in obs; cannot join labels, skipping")
            del subset_adata; gc.collect()
            continue

        subset_adata.obs = subset_adata.obs.set_index("cell_id", drop=False)
        subset_adata.obs = subset_adata.obs.join(df_cell[["celltype_update"]], how="left")

        # convert 'day' to ordered numeric category (for plotting/metadata), AFTER subsetting
        # keep original strings for selection logic above
        try:
            day_num = subset_adata.obs["day"].astype(str).str.extract(r"(\d+(?:\.\d+)?)").astype(float)[0]
            subset_adata.obs["day_num"] = day_num
            subset_adata.obs["day_num"] = pd.Categorical(subset_adata.obs["day_num"], ordered=True)
        except Exception as e:
            print(f"  -> warning: could not parse day to numeric: {e}")

        # --- run LR-OT methods + evaluation ---
        try:
            df_out = single_cell.run_pairwise_eval(
                subset_adata, t1, t2,
                methods=("frlc", "monge_conj", "lot" ),
                label_col="celltype_update",
                seed=0, use_pca=True, pca_key="X_pca", n_comps=50,
                max_rank=None,
                subsample_to_nonprime=subsample_to_nonprime_tp[tp],
                hiref_max_Q=max_Qs[tp], hiref_max_rank=max_ranks[tp]
                # or set to a fixed rank, e.g., 10
            )
            print(df_out)
            all_results.append(df_out)
        except Exception as e:
            print(f"  -> evaluation failed: {e}")

        # free memory aggressively
        del subset_adata; gc.collect()
    
    # close backed AnnData view (let file handle go)
    del adata; gc.collect()

# save combined results
if all_results:
    all_results = pd.concat(all_results, ignore_index=True)
    save_csv = "sc_timepair_lr_ot_results.csv"
    all_results.to_csv(save_csv, index=False)
    print("Saved:", save_csv)
else:
    print("No results produced.")


[intra] Aligning timepoint E8.5 (embryo_11) → E8.75 (embryo_14)
Iteration: 0
Optimized rank-annealing schedule: [27, 697]


2025-09-21 22:27:08.666 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:185 - Initialization Costs: (0.23417323061625728, 0.2345893174726128)


  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0        E8.5       E8.75        frlc    43  0.552525  0.556304  0.217489   
1        E8.5       E8.75  monge_conj    43  0.505576  0.639308  0.328732   
2        E8.5       E8.75         lot    43  0.520181  0.604899  0.282691   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  
0  0.531313  0.199441  0.525272    16.459865    18819  
1  0.616598  0.306974  0.722262    63.389580    18819  
2  0.591938  0.271684  0.610775     8.772032    18819  
[intra] Aligning timepoint E8.75 (embryo_14) → E9.0 (embryo_16)
Subsampled to n=30240 from n=30655 [needed for Hierarchical Refinement]
Iteration: 0
Optimized rank-annealing schedule: [315, 96]


2025-09-21 22:31:12.918 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:185 - Initialization Costs: (0.19839281499003175, 0.19872728933111744)


  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0       E8.75        E9.0        frlc    53  0.404843  0.533738  0.174067   
1       E8.75        E9.0  monge_conj    53  0.383924  0.596766  0.230686   
2       E8.75        E9.0         lot    53  0.390198  0.559111  0.193351   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  
0  0.541283  0.177729  0.491634    16.927181    30240  
1  0.597506  0.229977  0.544787   177.123930    30240  
2  0.566558  0.197148  0.486511    10.881868    30240  
[intra] Aligning timepoint E9.0 (embryo_16) → E9.25 (embryo_20)
Subsampled to n=45360 from n=45685 [needed for Hierarchical Refinement]
Iteration: 0
Optimized rank-annealing schedule: [504, 90]


2025-09-21 22:37:13.583 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:185 - Initialization Costs: (0.22055347770114303, 0.21956177622307665)
2025-09-21 22:37:26.492201: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 28.03GiB (30100401916 bytes) by rematerialization; only reduced to 30.70GiB (32966040378 bytes), down from 30.70GiB (32966766138 bytes) originally
2025-09-21 22:37:39.281368: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 13.54GiB (14542473292 bytes) by rematerialization; only reduced to 15.33GiB (16464431120 bytes), down from 15.33GiB (16464431120 bytes) originally
2025-09-21 22:37:39.294631: E external/xla/xla/service/gpu/gpu_hlo_schedule.cc:795] The byte size of input/output arguments (32920473608) exceeds the base limit (31804391424). This indicates an error in the calculation!
2025-09-21 22:37:39.294837: W external/xla/xla/hlo/transforms/simpl

  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0        E9.0       E9.25        frlc    57  0.481352  0.524199  0.158256   
1        E9.0       E9.25  monge_conj    57  0.451887  0.562633  0.190489   
2        E9.0       E9.25         lot    57       NaN       NaN       NaN   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  \
0  0.514501  0.155378  0.470642    19.314041  45360.0   
1  0.553779  0.186643  0.499502   286.953478  45360.0   
2       NaN       NaN       NaN          NaN      NaN   

                                               error  
0                                                NaN  
1                                                NaN  
2  RESOURCE_EXHAUSTED: Out of memory while trying...  
[intra] Aligning timepoint E9.25 (embryo_20) → E9.5 (embryo_24)
Subsampled to n=75600 from n=83059 [needed for Hierarchical Refinement]
Iteration: 0
Optimized rank-annealing schedule: [840, 90]


2025-09-21 22:46:58.808 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:185 - Initialization Costs: (0.1774943484188905, 0.17671990964548212)
2025-09-21 22:47:00.245181: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 28.07GiB (30137180790 bytes) by rematerialization; only reduced to 42.66GiB (45808117544 bytes), down from 42.66GiB (45808117544 bytes) originally
2025-09-21 22:47:10.664809: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 42.62GiB (rounded to 45767598080)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-09-21 22:47:10.734109: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] **_______________________________________________________________________________________

  timepoint_A timepoint_B      method  rank  ot_cost     A_AMI     A_ARI  \
0       E9.25        E9.5        frlc    67  0.43125  0.484287  0.128585   
1       E9.25        E9.5  monge_conj    67      NaN       NaN       NaN   
2       E9.25        E9.5         lot    67      NaN       NaN       NaN   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  \
0  0.488075  0.130265  0.440566    33.867036  75600.0   
1       NaN       NaN       NaN          NaN      NaN   
2       NaN       NaN       NaN          NaN      NaN   

                                               error  
0                                                NaN  
1  RESOURCE_EXHAUSTED: Out of memory while trying...  
2  RESOURCE_EXHAUSTED: Out of memory while trying...  
[intra] Aligning timepoint E9.5 (embryo_24) → E9.75 (embryo_28)



KeyboardInterrupt

